In [89]:
import numpy as np
import pandas as pd

In [ ]:
# Loading datasets
movies = pd.read_csv("./dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("./dataset/tmdb_5000_credits.csv")

Preprocessing - Start

In [91]:
# merging both movies and credits dataset into single dataset
movies = movies.merge(credits,on='title')
# display(movies.columns)

In [92]:
# important columns: genres, id, keywords, overview, tagline, title,vote_average,vote_count
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
# display(movies.head(1))

In [ ]:
#droping null values
movies.dropna(inplace=True)

In [94]:
import ast
def Extract_Keywords_From_Json_Data(obj):
    keywords = []
    for i in ast.literal_eval(obj):
        keywords.append(i["name"])
    return keywords


def Extract_Keywords_From_Cast_Column(obj):
    keywords = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            keywords.append(i["name"])
            counter+1
        else:
            break
    return keywords 


def Extract_Keywords_From_Crew_Column(obj):
    keywords = []
    for i in ast.literal_eval(obj):
        if i["job"]=='Director':
            keywords.append(i["name"])
            break
    return keywords 

In [95]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['genres'] = movies['genres'].apply(Extract_Keywords_From_Json_Data)
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(Extract_Keywords_From_Json_Data)
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(Extract_Keywords_From_Cast_Column)
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(Extract_Keywords_From_Crew_Column)
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])


In [96]:
# display(movies.head(5))

In [97]:
# adding new column "tags" to the movies DataFrame
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [98]:
new_df = movies[['movie_id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [ ]:
# display(new_df)

NLTK(Natural Lang ToolKit) - Stemming ( ex : walking -> walk, walked -> walk, walk -> walk)

In [100]:
from nltk.stem.porter import PorterStemmer

# stemming converts words to their root form
ps = PorterStemmer()

def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))     #stem is used to convert loving -> love, loved -> love, love -> love
    return " ".join(y)

In [101]:
# Stemming each words to their root form ( ex : loving -> love, loved -> love, love -> love)
new_df['tags'] = new_df['tags'].apply(stem)

sklearn - CounterVector (To Convert text to numerical data)

In [102]:
# converting text to numerical data using word frequency
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english') # stop words removes the connecting words like (a,an,and,are etc) 

vectors = cv.fit_transform(new_df['tags']).toarray()

Preprocessing - End